In [19]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

In [20]:
import pandas as pd

from src.utils.logger import get_logger
from src.data.db import get_db_connection

In [ ]:
load_dotenv()
logger = get_logger(__name__)

conn = None

try:
    logger.info("Start connecting to the database")
    conn = get_db_connection()
    logger.info("Connected to the database successfully")

    select_query = "SELECT * FROM clean_job_posting_v2"

    logger.info("Fetching the query into a DataFrame")
    df = pd.read_sql(select_query, conn)
    logger.info("Data loaded successfully")

except Exception as e:
        logger.error(f" failed: {e}")

finally:
    if conn is not None:
        conn.close()
        logger.info("Database connection closed")        

2026-05-06 22:15:22 | INFO | __main__ | Start connecting to the database
2026-05-06 22:15:22 | INFO | src.data.db | Connecting to PostgreSQL
2026-05-06 22:15:22 | ERROR | src.data.db | Failed to connect to PostgreSQL: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
2026-05-06 22:15:22 | ERROR | __main__ |  failed: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'


# 1. Initial Data Loading and Structural Inspection

## 1.1 Dataset Preview

In [18]:
df.head()

NameError: name 'df' is not defined

## 1.2 Dataset Shape

In [ ]:
df.shape

(54, 9)

## 1.3 Column Names

In [ ]:
df.columns.tolist()

['standardized_job_title',
 'clean_job_title',
 'company_name',
 'job_country',
 'employment_type',
 'is_remote',
 'job_description',
 'job_source',
 'posted_at_utc']

## 1.4 Data Types

In [ ]:
df.dtypes

standardized_job_title                 object
clean_job_title                        object
company_name                           object
job_country                            object
employment_type                        object
is_remote                                bool
job_description                        object
job_source                             object
posted_at_utc             datetime64[ns, UTC]
dtype: object

## 1.5 DataFrame Info

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   standardized_job_title  54 non-null     object             
 1   clean_job_title         54 non-null     object             
 2   company_name            54 non-null     object             
 3   job_country             54 non-null     object             
 4   employment_type         54 non-null     object             
 5   is_remote               54 non-null     bool               
 6   job_description         54 non-null     object             
 7   job_source              54 non-null     object             
 8   posted_at_utc           34 non-null     datetime64[ns, UTC]
dtypes: bool(1), datetime64[ns, UTC](1), object(7)
memory usage: 3.6+ KB


## 1.6 Dataset Describe

In [ ]:
df.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max
standardized_job_title,54,2,data scientist,28,NaN,NaN,NaN,NaN,NaN,NaN
clean_job_title,54,42,data analyst,6,NaN,NaN,NaN,NaN,NaN,NaN
company_name,54,47,First Abu Dhabi Bank,2,NaN,NaN,NaN,NaN,NaN,NaN
job_country,54,3,QA,19,NaN,NaN,NaN,NaN,NaN,NaN
employment_type,54,3,full-time,51,NaN,NaN,NaN,NaN,NaN,NaN
is_remote,54,1,False,54,NaN,NaN,NaN,NaN,NaN,NaN
job_description,54,54,SUMMARY OF FUNCTIONS:\n\nThe Data Scientist su...,1,NaN,NaN,NaN,NaN,NaN,NaN
job_source,54,26,bebee,10,NaN,NaN,NaN,NaN,NaN,NaN
posted_at_utc,34,NaN,NaN,NaN,2026-04-19 18:19:24.705882368+00:00,2026-03-31 00:00:00+00:00,2026-04-19 00:00:00+00:00,2026-04-22 00:00:00+00:00,2026-04-23 00:00:00+00:00,2026-04-26 23:00:00+00:00


# 2. Data Quality Assessment

## 2.1 Missing Values Analysis

### Observations

- Most columns do not contain missing values.
- `posted_at_utc` contains missing values (~37%).
- Missing datetime values may affect future time-series and trend analysis.
- `job_description` has no missing values, which is important for the NLP phase.

In [ ]:
null_values = df.isnull().sum()
percentage_null = (df.isnull().sum() / df.shape[0]) * 100

missing_values_summary = pd.DataFrame({
    'missing_count': null_values,
    'missing_percentage': percentage_null
})

missing_values_summary


,missing_count,missing_percentage
standardized_job_title,0,0.000000
clean_job_title,0,0.000000
company_name,0,0.000000
job_country,0,0.000000
employment_type,0,0.000000
is_remote,0,0.000000
job_description,0,0.000000
job_source,0,0.000000
posted_at_utc,20,37.037037


## 2.2 Duplicate Records Analysis

### Observations

- No duplicate rows were detected in the dataset.
- This indicates that the data cleaning or ingestion process successfully avoided exact duplicate records.
- As a result, there is no immediate need for duplicate removal at this stage.

In [ ]:
print(df.duplicated().sum())

0


## 2.3 Data Type Optimization

### Observations

- Converted low-cardinality columns to categorical type.
- This improves memory efficiency and performance for grouping operations.
- Text-heavy and high-cardinality columns were kept as object type.

In [ ]:
cols = ['standardized_job_title', 'job_country', 'employment_type', 'job_source']
df[cols] = df[cols].astype('category')

# 3. Univariate Analysis

## 3.1 Job Role Distribution

### Observations

- The most frequent role is: data scientist (28 records)
- The dataset is relatively balanced between the two roles
- This indicates that both roles are reasonably represented, which is beneficial for fair analysis and future modeling

In [ ]:
df['standardized_job_title'].value_counts()

standardized_job_title
data scientist    28
data analyst      26
Name: count, dtype: int64

## 3.2 Job Country Distribution

### Observations

- The country with the highest number of job postings is Qatar (19 records)
- The distribution of job postings across the three countries is relatively balanced
- This indicates that all countries are reasonably represented, which is beneficial for fair analysis and future modeling

In [ ]:
df['job_country'].value_counts()

job_country
QA    19
SA    18
AE    17
Name: count, dtype: int64

## 3.3 Employment Type Distribution

### Observations

- The most common employment type is full-time (51 records)
- The distribution is highly skewed toward full-time roles, with very few internship (2) and contractor (1) positions
- This indicates that the dataset is strongly dominated by full-time opportunities, which may limit analysis for other employment types

In [ ]:
df['employment_type'].value_counts()

employment_type
full-time     51
internship     2
contractor     1
Name: count, dtype: int64

## 3.4 Remote Work Distribution

### Observations

- All job postings are marked as non-remote (False)
- This results in a fully skewed distribution toward on-site roles
- This is likely unrealistic and may indicate an issue in the data collection or preprocessing stage
- Further investigation is needed to verify whether remote job information is missing or incorrectly labeled

In [ ]:
df['is_remote'].value_counts()

is_remote
False    54
Name: count, dtype: int64

## 3.5 Job Description Quality Check

### Observations

- Job descriptions are present for all records (no missing values)
- The descriptions are generally detailed and contain relevant information such as responsibilities, required skills, and qualifications
- The text appears sufficiently rich and structured, making it suitable for NLP tasks such as skill extraction
- No obvious noise or extremely short descriptions were observed

In [ ]:
pd.set_option('display.max_colwidth',None)

print(f"Missing values:", df['job_description'].isnull().sum())
print(f"Total rows :", len(df['job_description']))

df['job_description'].head(5)

Missing values: 0
Total rows : 54


0                                                                                                                                                                                                         SUMMARY OF FUNCTIONS:\n\nThe Data Scientist supports the Office of Vice-Chancellor for Institutional Planning and Effectiveness by developing descriptive, predictive and prescriptive models using machine learning algorithms, conducting statistical analyses to support decision-making for higher management, and providing recommendations, based on data analytics, to enhance academic quality, student success, and performance effectiveness of colleges/offices .\n\nESSENTIAL DUTIES & RESPONSIBILITIES:\n\n•\nData Analysis / Data Engineering: Collect, clean, and preprocess data for analysis.\n\n•\nML Modeling: Develop predictive and prescriptive models using machine learning algorithms.\n\n•\nStatistical Analysis: Conduct statistical analyses to support decision-making and solve business problem

# 4. Bivariate Analysis

## 4.1 Job Role Distribution by Country

### Observations

- Job roles are relatively evenly distributed across all countries
- Within each country, the number of Data Scientists and Data Analysts is very close, with no significant dominance of one role
- Saudi Arabia shows a perfectly balanced distribution between the two roles
- Qatar has a slight preference for Data Scientist roles, but the difference is minimal
- Overall, there is no strong country-specific specialization for a particular role in this dataset

In [ ]:
print("\nJob Distribution over the countries:")
print(df.groupby('job_country')['standardized_job_title'].value_counts().sort_values(ascending=False))


Job Distribution over the countries:
job_country  standardized_job_title
QA           data scientist            10
AE           data scientist             9
SA           data analyst               9
QA           data analyst               9
SA           data scientist             9
AE           data analyst               8
Name: count, dtype: int64


C:\Users\anton\AppData\Local\Temp\ipykernel_8132\758884820.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('job_country')['standardized_job_title'].value_counts().sort_values(ascending=False))


## 4.2 Employment Type by Job Role

### Observations

- Full-time employment is the dominant type for both Data Analyst and Data Scientist roles
- Data Analyst roles in this dataset are exclusively full-time, with no internship or contractor positions observed
- Data Scientist roles show more variety, including full-time, internship, and contractor positions
- This suggests that Data Scientist roles may offer more diverse employment types, although this may be influenced by the limited dataset size

In [ ]:
print("\nEmployment type by JOb Role: ")
print(df.groupby('standardized_job_title')['employment_type'].value_counts())


Employment type by JOb Role: 
standardized_job_title  employment_type
data analyst            full-time          26
                        contractor          0
                        internship          0
data scientist          full-time          25
                        internship          2
                        contractor          1
Name: count, dtype: int64


C:\Users\anton\AppData\Local\Temp\ipykernel_8132\1503802845.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('standardized_job_title')['employment_type'].value_counts())


## 4.3 Remote Work Distribution by Job Role

### Observations

- All job postings for both Data Analyst and Data Scientist roles are marked as non-remote (on-site)
- There is no variation in remote work across roles, indicating that neither role appears to offer remote opportunities in this dataset
- This result is likely unrealistic and may indicate a limitation or issue in the data collection or preprocessing process

In [ ]:
print("\nEmployment type by JOb Role: ")
print(df.groupby('standardized_job_title')['is_remote'].value_counts())


Employment type by JOb Role: 
standardized_job_title  is_remote
data analyst            False        26
data scientist          False        28
Name: count, dtype: int64


C:\Users\anton\AppData\Local\Temp\ipykernel_8132\4274660101.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('standardized_job_title')['is_remote'].value_counts())


# 5. Preparing for NLP Analysis

## 5.1 Identifying Relevant Text Columns for NLP

### Selected Column for NLP

- job_description

### Justification

- The column contains no missing values, ensuring complete coverage across all records
- It includes rich and detailed textual information such as responsibilities, required skills, and qualifications
- The content is directly relevant to the business objective of extracting skills and tools from job postings
- Based on EDA, the text quality is sufficient for downstream NLP tasks

## 5.2 Potential Challenges in NLP Processing

### Potential Challenges

- Inconsistent representation of skills (e.g., Python, python, PYTHON), which may require normalization
- Job descriptions are unstructured, with skills embedded within paragraphs rather than listed explicitly
- Presence of noise or irrelevant text (e.g., company descriptions or general statements) that may interfere with accurate skill extraction

## 5.3 Defining the NLP Objective

### NLP Objective

- Extract key technical skills, tools, and technologies mentioned in job descriptions
- Associate these extracted skills with each standardized job role (e.g., Data Analyst, Data Scientist)
- Examples include programming languages (Python, SQL, R), tools (Power BI, Tableau), and technologies (machine learning frameworks)

### Business Purpose

- Enable identification of the most in-demand skills for each job role
- Help job seekers understand which skills and tools they need to learn for specific roles
- Assist recruiters in identifying the key competencies required for each role
- Support the development of a data-driven dashboard that summarizes job market requirements

## 5.4 Initial Skill Extraction Approach

### Skill Extraction Approach

- Use a dictionary-based method where we define a list of known skills, tools, and technologies
- The extraction process will search for these predefined skills within the job descriptions

### Why This Approach

- It is simple and easy to implement without requiring complex NLP models
- It provides full control over what is being extracted
- It is easy to debug and validate results
- It is suitable for the current dataset size and project stage

## 5.5 Initial Skill Dictionary

### Defined Skill Categories

#### Programming Languages
- Python
- SQL
- R

#### Data Visualization Tools
- Power BI
- Tableau
- Excel
- Google Sheets

#### Technologies and Concepts
- Machine Learning
- Artificial Intelligence
- Deep Learning
- LLM (Large Language Models)
- NLP (Natural Language Processing)

### Observations

- The initial dictionary focuses on common and widely used skills, tools, and technologies relevant to Data Analyst and Data Scientist roles
- The selected items represent a foundational set of competencies frequently mentioned in job descriptions
- This dictionary will be iteratively expanded and refined based on extraction results and newly observed skills in the dataset

In [ ]:
Data_skills = ['Python', 'SQL', 'R']
Data_tools = ['PowerBI', 'Tableau', 'Excell', 'google sheets']
Data_tech = ['Machine Learning', 'Deep Learning', 'NLP'] 

# 6. NLP Skill Extraction Implementation

## 6.1 Text Normalization

### Why Text Normalization is Important

- Skills may appear in different formats such as uppercase, lowercase, or mixed case (e.g., Python, python, PYTHON)
- Without normalization, the extraction process may treat the same skill as different values
- Converting text to lowercase helps standardize the text and improves the consistency and accuracy of skill matching

## 6.1.1 Creating a Normalized Text Column

In [ ]:
df['Standarized_job_descripton'] = df['job_description'].lower()

NameError: name 'df' is not defined